# Corrected CAFE batch on Colab GPU

Run this notebook on a **T4 GPU** (`Runtime → Change runtime type → T4 GPU`). It runs a two-video equivalence check first and starts the complete batch only if labels, candidate/cue/interval selections, controls, and decisions match the validated corrected JSONs. Numeric values must satisfy `abs(actual - reference) <= 1e-4 + 1e-4 * abs(reference)`.

Before starting, place these assets in Google Drive:
- `MyDrive/cafe_phase16_assets.zip` (the existing cache/checkpoint/detector bundle)
- `MyDrive/CAFE/dataset/` with the 60 manifest videos (or ensure the bundle/cache already covers every video)
- The two validated files `Deepfakes_000_003__real.json` and `real_011__real.json` in `MyDrive/CAFE/results/batch_corrected/`

The notebook clones the repository, installs project dependencies without replacing Colab's CUDA-enabled PyTorch, extracts the existing asset bundle, and stores all corrected results on Drive. Rerun the full-batch cell after a disconnect to resume; successful per-run JSONs are kept and skipped.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; ensure Drive assets are mounted manually.')

REPO_URL = 'https://github.com/VINYTHA1708/CAFE.git'
REPO_DIR = Path('/content/CAFE')
DRIVE_CAFE_DIR = Path('/content/drive/MyDrive/CAFE')
runner_path = REPO_DIR / 'scripts' / 'run_batch_corrected_colab.py'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
if not runner_path.is_file():
    uploaded_runner = DRIVE_CAFE_DIR / 'scripts' / 'run_batch_corrected_colab.py'
    if uploaded_runner.is_file():
        runner_path.parent.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy2(uploaded_runner, runner_path)
    else:
        raise FileNotFoundError(f'{runner_path} missing. Either use a repository revision containing the new runner or upload it to {uploaded_runner}.')
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

ASSET_ARCHIVE = Path('/content/drive/MyDrive/cafe_phase16_assets.zip')
DRIVE_DATASET_DIR = DRIVE_CAFE_DIR / 'dataset'
OUTPUT_DIR = DRIVE_CAFE_DIR / 'results' / 'batch_corrected'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO_DIR)
print('Persistent corrected outputs:', OUTPUT_DIR)

## Install dependencies and restore assets

The dependency step reads the repository's pinned requirements but preserves Colab's preinstalled `torch`/`torchvision` CUDA build. The phase-16 asset archive contains cached preprocessing/landmarks, the frozen checkpoint, and detector reference files; it does not contain the source videos.

In [ ]:
import importlib.metadata
import re
import tempfile
import zipfile

requirements_path = REPO_DIR / 'requirements.txt'
raw = requirements_path.read_bytes()
requirements_text = raw.decode('utf-16' if b'\x00' in raw[:100] else 'utf-8-sig')
filtered = []
for line in requirements_text.splitlines():
    stripped = line.strip()
    if not stripped or stripped.startswith('#'):
        continue
    package = re.split(r'[<=>!~;\[]', stripped, maxsplit=1)[0].strip().lower().replace('_', '-')
    if package in {'torch', 'torchvision'}:
        continue
    filtered.append(stripped)
with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as req_file:
    req_file.write('\n'.join(filtered) + '\n')
    filtered_requirements = req_file.name
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', filtered_requirements], check=True)
print('Installed repository dependencies; preserving Colab torch/torchvision.')

required_asset_paths = [
    REPO_DIR / 'detector_reference',
    REPO_DIR / 'models' / 'EfficientNetB4_FFPP' / 'EfficientNetB4_FFPP_bestval.pth',
    REPO_DIR / 'models' / 'face_landmarks' / 'face_landmarker.task',
]
if not all(path.exists() for path in required_asset_paths):
    if not ASSET_ARCHIVE.is_file():
        raise FileNotFoundError(f'Missing model/detector assets and archive: {ASSET_ARCHIVE}')
    with zipfile.ZipFile(ASSET_ARCHIVE) as archive:
        root = REPO_DIR.resolve()
        for member in archive.infolist():
            destination = (REPO_DIR / member.filename).resolve()
            if destination != root and root not in destination.parents:
                raise ValueError(f'Unsafe path in asset archive: {member.filename}')
        archive.extractall(REPO_DIR)
    print('Extracted phase-16 cached data, model and detector assets.')

dataset_link = REPO_DIR / 'dataset'
if not dataset_link.exists() and DRIVE_DATASET_DIR.is_dir():
    dataset_link.symlink_to(DRIVE_DATASET_DIR, target_is_directory=True)
print('Dataset path:', dataset_link, 'exists:', dataset_link.exists())

In [ ]:
import json
import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime before proceeding.')
print('GPU:', torch.cuda.get_device_name(0))

checkpoint = REPO_DIR / 'models' / 'EfficientNetB4_FFPP' / 'EfficientNetB4_FFPP_bestval.pth'
assert checkpoint.is_file(), f'Missing frozen EfficientNet-B4 checkpoint: {checkpoint}'
manifest = pd.read_csv(REPO_DIR / 'data' / 'manifest.csv')
assert len(manifest) == 60 and manifest.video_id.nunique() == 60, 'Expected the original 60-video manifest.'
print('Checkpoint:', checkpoint, f'({checkpoint.stat().st_size:,} bytes)')
print('Manifest:', len(manifest), 'videos;', manifest.label.value_counts().to_dict())

from scripts.run_batch_corrected_colab import (
    LEFT_EYE_INDICES, MOUTH_INDICES, RIGHT_EYE_INDICES,
    corrected_region_mask,
)
from cafe.landmarks import region_mask as production_region_mask
assert RIGHT_EYE_INDICES == (7, 33, 133, 144, 145, 153, 154, 155, 157, 158, 159, 160, 161, 163, 173, 246)
assert LEFT_EYE_INDICES == (249, 263, 362, 373, 374, 380, 381, 382, 384, 385, 386, 387, 388, 390, 398, 466)
assert MOUTH_INDICES == (0, 13, 14, 17, 37, 39, 40, 61, 78, 80, 81, 82, 84, 87, 88, 91, 95, 146, 178, 181, 185, 191, 267, 269, 270, 291, 308, 310, 311, 312, 314, 317, 318, 321, 324, 375, 402, 405, 409, 415)
for video_id in ('Deepfakes_000_003', 'real_011'):
    cache = REPO_DIR / 'cache' / video_id
    landmarks = np.load(cache / 'landmarks.npy', mmap_mode='r')
    sample = next(frame for frame in landmarks if np.isfinite(frame).all())
    eye_mask = corrected_region_mask(sample, 'eyes', feather_px=5)
    mouth_mask = corrected_region_mask(sample, 'mouth', feather_px=5)
    old_face_mask = production_region_mask(sample, 'face', feather_px=5)
    new_face_mask = corrected_region_mask(sample, 'face', feather_px=5)
    assert eye_mask.sum() > 0 and mouth_mask.sum() > 0
    assert np.array_equal(old_face_mask, new_face_mask), 'face_texture mask changed'
    print(video_id, 'eye-mask area=', round(float(eye_mask.mean()), 5), 'mouth-mask area=', round(float(mouth_mask.mean()), 5), 'face_texture unchanged=True')

missing = []
for row in manifest.itertuples(index=False):
    cache = REPO_DIR / 'cache' / row.video_id
    cached = all((cache / name).is_file() for name in ('faces.npy', 'landmarks.npy', 'index.json'))
    source = REPO_DIR / row.path
    if not cached and not source.is_file():
        missing.append(row.video_id)
if missing:
    raise FileNotFoundError(f'Missing both complete cached data and source video for: {missing}')
print('All 60 manifest entries have cached inputs or source videos.')

baseline_dir = OUTPUT_DIR
for video_id in ('Deepfakes_000_003', 'real_011'):
    baseline = baseline_dir / f'{video_id}__real.json'
    if not baseline.is_file():
        raise FileNotFoundError(f'Upload the validated corrected reference JSON to Drive: {baseline}')
print('Validated two-video reference JSONs found.')

## GPU equivalence gate

This writes the trial to a unique validation subdirectory and does not replace the validated reference JSONs. It requires exact prediction, candidate count/order, selected cues, intervals, control specifications, p-values, support decisions, and close numeric values. Any mismatch stops the notebook before the full run.

In [ ]:
from datetime import datetime
import math

ATOL = 1e-4
RTOL = 1e-4
validation_dir = OUTPUT_DIR / ('_gpu_validation_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
subprocess.run([
    sys.executable, 'scripts/run_batch_corrected_colab.py',
    '--videos', 'Deepfakes_000_003', 'real_011',
    '--conditions', 'real',
    '--out', str(validation_dir),
    '--batch-size', '16',
    '--require-cuda',
], cwd=REPO_DIR, check=True)

def compare_tree(reference, actual, path='$', differences=None, numeric=None):
    if differences is None:
        differences = []
    if numeric is None:
        numeric = {'checked': 0, 'max_abs_diff': 0.0}
    if path == '$.timings':
        return differences, numeric
    if isinstance(reference, bool) or isinstance(actual, bool):
        if type(reference) is not type(actual) or reference != actual:
            differences.append(f'{path}: {reference!r} != {actual!r}')
    elif isinstance(reference, (int, float)) and isinstance(actual, (int, float)):
        numeric['checked'] += 1
        delta = abs(float(reference) - float(actual))
        numeric['max_abs_diff'] = max(numeric['max_abs_diff'], delta)
        if not math.isclose(float(reference), float(actual), abs_tol=ATOL, rel_tol=RTOL):
            differences.append(f'{path}: {reference!r} != {actual!r} (abs diff {delta:.3g})')
    elif isinstance(reference, dict) and isinstance(actual, dict):
        if reference.keys() != actual.keys():
            differences.append(f'{path}: object keys differ')
        for key in reference.keys() & actual.keys():
            compare_tree(reference[key], actual[key], f'{path}.{key}', differences, numeric)
    elif isinstance(reference, list) and isinstance(actual, list):
        if len(reference) != len(actual):
            differences.append(f'{path}: list lengths differ ({len(reference)} vs {len(actual)})')
        for index, (left, right) in enumerate(zip(reference, actual)):
            compare_tree(left, right, f'{path}[{index}]', differences, numeric)
    elif reference != actual:
        differences.append(f'{path}: {reference!r} != {actual!r}')
    return differences, numeric

validation_report = {}
for video_id in ('Deepfakes_000_003', 'real_011'):
    reference = json.loads((baseline_dir / f'{video_id}__real.json').read_text(encoding='utf-8'))
    actual = json.loads((validation_dir / f'{video_id}__real.json').read_text(encoding='utf-8'))
    differences, numeric = compare_tree(reference, actual)
    validation_report[video_id] = {'match': not differences, **numeric, 'differences': differences[:10]}
print(json.dumps({'tolerance': {'absolute': ATOL, 'relative': RTOL}, 'results': validation_report}, indent=2))
if not all(result['match'] for result in validation_report.values()):
    raise RuntimeError('GPU validation did not match the previously validated results; the full batch was NOT started.')
print('GPU equivalence gate passed. Full run is now permitted.')

## Run or resume the full 60 × 3 experiment

This cell writes each completed video-condition JSON atomically and refreshes the summary CSV after every run. Progress includes an ETA. On disconnect, reconnect the same Drive, rerun setup and validation, then rerun this cell; successful outputs are resumed rather than overwritten.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/run_batch_corrected_colab.py',
    '--manifest', 'data/manifest.csv',
    '--config', 'config.yaml',
    '--out', str(OUTPUT_DIR),
    '--conditions', 'real', 'placebo', 'authentic',
    '--batch-size', '16',
    '--require-cuda',
], cwd=REPO_DIR, check=True)

summary_path = OUTPUT_DIR / 'batch_corrected_summary.csv'
summary = pd.read_csv(summary_path)
print('Corrected summary:', summary_path)
print('Candidate rows:', len(summary), 'run JSONs:', len(list(OUTPUT_DIR.glob('*__*.json'))))
print('Statuses:', summary.groupby('status').size().to_dict())
display(summary.head())